## Analyze WH events

This script summarizes the event structures in the W-H-MEEG dataset
for consistency and potential remapping.  The script assumes that
an extra set of event files `_events_temp.tsv` has been dumped
from the `.set` MEEG files so that consistency of the two event
representation can be checked.

In [ ]:
bids_root_path = 'G:/WH_working'

In [ ]:
from hed.tools.io_utils import get_file_list

event_files_bids = get_file_list(bids_root_path, extensions=[".tsv"], name_suffix="_events")
event_files_eeg = get_file_list(bids_root_path, extensions=[".tsv"], name_suffix="_events_temp")
bids_skip = ['onset', 'duration', 'sample', 'response_time', 'stim_file', 'HED']
eeg_skip = ['latency', 'urevent', 'sample_offset', 'stim_file']

We print out the column names of each event file to make sure uniform.

In [ ]:
import os
from hed.tools.data_utils import get_new_dataframe

bids_file_dict = {}
print(f"\nBIDS form of the events: {len(event_files_bids)}")
for file in event_files_bids:
    base = os.path.basename(file)
    pieces = base.split('_')
    key = f"{pieces[0]}_{pieces[-2]}"
    df = get_new_dataframe(file)
    bids_file_dict[key] = file
    print(f"{key}: {str(list(df.columns.values))}")

In [ ]:
eeg_file_dict = {}
print(f"\nEEG form of the events: {len(event_files_eeg)} files")
for file in event_files_eeg:
    base = os.path.basename(file)
    pieces = base.split('_')
    key = f"{pieces[0]}_{pieces[-3]}"
    df = get_new_dataframe(file)
    eeg_file_dict[key] = file
    print(f"{key}: {str(list(df.columns.values))}")

In [ ]:
from hed.tools.col_dict import ColumnDict
bids_dicts_all = ColumnDict(skip_cols=bids_skip, name=f"{bids_root_path} BIDS" )
bids_dicts = {}
for key, file in bids_file_dict.items():
    orig_dict = ColumnDict(skip_cols=bids_skip, name=f"{file} BIDS")
    orig_dict.update(file)
    bids_dicts[key] = orig_dict
    bids_dicts_all.update_dict(orig_dict)
bids_dicts_all.print()

In [ ]:
from hed.tools.col_dict import ColumnDict
eeg_dicts_all = ColumnDict(skip_cols=eeg_skip, name=f"{bids_root_path} EEG" )
eeg_dicts = {}
for key, file in eeg_file_dict.items():
    eeg_dict = ColumnDict(skip_cols=eeg_skip, name=f"{file} EEG")
    eeg_dict.update(file)
    eeg_dicts[key] = eeg_dict
    eeg_dicts_all.update_dict(eeg_dict)
eeg_dicts_all.print()

Make the following changes to the BIDS events files:

1. Rename `trial_lag` as `rep_lag`.
2. Rename `repetition_type` as `rep_type`.
3. Rename `trigger` as `value`.
4. Remove the first data row of each file.
5. Rename `right_sym` in new first row of `event_type` column to be `setup_right_sym`.
6. Rename `left_sym` in new first row of `event_type` column to be `setup_left_sym`.
7. Save as `_events_temp2.tsv`

In [ ]:
bids_file_dict = {}
print(f"\nBIDS form of the events: {len(event_files_bids)}")
for my_file in event_files_bids:
    print(my_file)
    df = get_new_dataframe(my_file)
    df.rename(columns={"trial_lag": "rep_lag", "repetition_type": "rep_status", "trigger":"value"}, inplace=True)
    df.drop([0], inplace=True)

    if  df.loc[df.index[0], 'event_type']== 'right_sym':
         df.loc[df.index[0], 'event_type'] = 'setup_right_sym'
    elif  df.loc[df.index[0], 'event_type']== 'left_sym':
         df.loc[df.index[0], 'event_type'] = 'setup_right_sym'
    else:
        print(f"...bad first line in {my_file}")
    filename = my_file[:-4] + "_temp2.tsv"
    print(filename)
    df.to_csv(filename, sep='\t', index=False)

In [ ]:
#os.remove("demofile.txt")

# remove_list = get_file_list(bids_root_path, extensions=[".tsv"], suffix="._temp2")
# for file in remove_list:
#     os.remove(file)
#     print(file)